# Bangalore Marketing ROI Report
### Rs. 6,00,000 Budget | 25 Campaigns | Data Analyst Track

---

**Objective:** Analyse the performance of 25 marketing campaigns across Bangalore zones using:
- **Z-Score analysis** to flag failed campaigns
- **Correlation heatmap** to identify the most effective channel drivers
- **Subplot dashboard** for an executive ROI forecast view
- A **Strategic Recommendations** box derived from the data


## 1. Setup & Imports

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Notebook display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
print("Libraries loaded successfully.")


## 2. Load Campaign Data

The tracker dataset `bangalore_campaigns.csv` contains **25 campaigns** with realistic
CPC and conversion rates per channel, all summing to the Rs. 6,00,000 budget.


In [ ]:
DATA_FILE = Path('../data/bangalore_campaigns.csv').resolve()
df = pd.read_csv(DATA_FILE)
print(f"Shape: {df.shape}  |  Total Spend: Rs. {df['Spend'].sum():,.0f}")
df.head(10)


## 3. Dataset Overview

In [ ]:
print("--- Column Info ---")
df.info()
print()
print("--- Descriptive Stats ---")
df.describe().T


## 4. Z-Score Analysis — Identifying Failed Campaigns

> A campaign is classified as **Failed** if its ROI Z-Score < -1.0  
> (i.e. its ROI is more than 1 standard deviation below the mean).


In [ ]:
failed = df[df['Status'] == 'Failed'][['Campaign_ID', 'Channel', 'Location', 'Spend', 'ROI_Percentage', 'ROI_Z_Score']]
print(f"Total Failed Campaigns : {len(failed)}")
print(f"Budget at Risk         : Rs. {failed['Spend'].sum():,.0f}")
print()
failed


In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
bar_colors = ['#e74c3c' if z < -1.0 else '#3498db' for z in df['ROI_Z_Score']]
ax.bar(df['Campaign_ID'], df['ROI_Percentage'], color=bar_colors, edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
ax.set_title('Campaign ROI % with Z-Score Failure Detection', fontsize=14, fontweight='bold')
ax.set_xlabel('Campaign ID'); ax.set_ylabel('ROI (%)')
ax.tick_params(axis='x', rotation=45)
ax.spines[['top','right']].set_visible(False)
ax.legend(handles=[
    mpatches.Patch(color='#3498db', label='Active'),
    mpatches.Patch(color='#e74c3c', label='Failed (Z < -1.0)'),
], loc='upper right')
plt.tight_layout()
plt.show()


## 5. Correlation Heatmap — Channel Effectiveness Drivers

Which metric actually drives ROI? The heatmap surfaces linear relationships
between Spend, Clicks, Conversions, Revenue, and ROI%.


In [ ]:
corr_cols = ['Spend', 'Clicks', 'Conversions', 'Revenue', 'ROI_Percentage']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=1, linecolor='white', annot_kws={'size':11,'weight':'bold'},
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap: Marketing Metrics', fontsize=13, fontweight='bold', pad=12)
ax.tick_params(axis='x', rotation=30); ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Strongest correlations with ROI
print("Pearson Correlations with ROI_Percentage:")
print(corr['ROI_Percentage'].sort_values(ascending=False).to_string())


## 6. Channel-Level Performance Summary

In [ ]:
channel_summary = df.groupby('Channel').agg(
    Campaigns   = ('Campaign_ID', 'count'),
    Total_Spend = ('Spend', 'sum'),
    Avg_ROI     = ('ROI_Percentage', 'mean'),
    Total_Revenue = ('Revenue', 'sum'),
).round(2).sort_values('Avg_ROI', ascending=False)
channel_summary['Avg_ROI'] = channel_summary['Avg_ROI'].map('{:.1f}%'.format)
channel_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ch_roi = df.groupby('Channel')['ROI_Percentage'].mean().sort_values(ascending=False)
bars = ax.bar(ch_roi.index, ch_roi.values, color='#3498db', edgecolor='white')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Average ROI% by Channel', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg ROI (%)'); ax.tick_params(axis='x', rotation=25)
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, ch_roi.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, f'{val:.0f}%',
            ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()


## 7. ROI Forecast — Spend vs ROI (Linear Regression)

A linear model fits the relationship between spend amount and ROI percentage.
Bubble size = number of conversions.


In [ ]:
fc_spend = np.linspace(df['Spend'].min(), df['Spend'].max(), 200)
slope, intercept, r_val, p_val, _ = stats.linregress(df['Spend'], df['ROI_Percentage'])
fc_roi   = intercept + slope * fc_spend

print(f"Slope     : {slope:.4f}  (ROI change per Rs. 1 spend)")
print(f"Intercept : {intercept:.2f}%")
print(f"R-squared : {r_val**2:.4f}")
print(f"p-value   : {p_val:.4f}")


In [ ]:
palette = {
    'Google Ads':            '#3498db',
    'Instagram Ads':         '#e91e63',
    'Facebook Ads':          '#4267B2',
    'LinkedIn Ads':          '#0077b5',
    'Influencer Marketing':  '#9b59b6',
    'Outdoor (Billboards)':  '#e67e22',
}

fig, ax = plt.subplots(figsize=(10, 5))
for ch, grp in df.groupby('Channel'):
    ax.scatter(grp['Spend'], grp['ROI_Percentage'],
               s=grp['Conversions'].clip(lower=5)*4+40,
               color=palette.get(ch, '#888'), alpha=0.85,
               edgecolors='white', linewidth=0.7, label=ch, zorder=4)

ax.plot(fc_spend, fc_roi, color='#e74c3c', linewidth=2, linestyle='--',
        label=f'Trend (r={r_val:.2f})')
ax.set_title('ROI Forecast & Channel Effectiveness', fontsize=13, fontweight='bold')
ax.set_xlabel('Spend (Rs.)'); ax.set_ylabel('ROI (%)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()


## 8. Full Executive Dashboard

In [ ]:
# Run the dashboard script to regenerate the PNG
import subprocess, sys
result = subprocess.run(
    [sys.executable, Path('../scripts/marketing_roi_dashboard.py').resolve()],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


In [ ]:
from IPython.display import Image
Image(filename=Path('../docs/bangalore_marketing_dashboard.png').resolve(), width=1100)


## 9. Strategic Recommendations

Based on the full analysis above:


In [ ]:
best_channel  = df.groupby('Channel')['ROI_Percentage'].mean().idxmax()
worst_channel = df.groupby('Channel')['ROI_Percentage'].mean().idxmin()
best_roi      = df.groupby('Channel')['ROI_Percentage'].mean().max()
failed_spend  = df[df['Status'] == 'Failed']['Spend'].sum()
failed_count = len(df[df['Status'] == 'Failed'])

print("=" * 65)
print("  STRATEGIC RECOMMENDATIONS — Bangalore Campaign Analysis")
print("=" * 65)
print(f"  1. CUT LOSSES     | {failed_count} campaigns failed (Z < -1.0).")
print(f"                    | Rs. {failed_spend:,.0f} is recoverable budget.")
print(f"  2. SCALE UP       | '{best_channel}' yields avg ROI of {best_roi:.0f}%.")
print(f"                    | Increase allocation here next quarter.")
print(f"  3. INVESTIGATE    | '{worst_channel}' consistently underperforms.")
print(f"                    | Pause it and audit creatives / audience targeting.")
print(f"  4. KEY DRIVER     | Conversions correlate strongest with Revenue.")
print(f"                    | Optimise conversion rate over click volume.")
print("=" * 65)
